In [ ]:
# Install required libraries
!pip -q install sentence-transformers faiss-cpu transformers torch

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

# 1. Knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# 2. Load embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Create document embeddings
doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True
).astype("float32")

# 4. Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# 5. User query
query = "What is RAG in AI?"

# 6. Encode query
query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

# 7. Retrieve top-2 most similar documents
D, I = index.search(query_embedding, k=2)

retrieved_chunks = [documents[i] for i in I[0]]

# 8. Build prompt
context = " ".join(retrieved_chunks)
prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

# 9. Load text generation model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

# 10. Generate answer
answer = generator(prompt, max_length=60, do_sample=False)

# 11. Display results
print("Retrieved Context:")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}")

print("\nGenerated Answer:")
print(answer[0]["generated_text"])
